# CNN 实验：从 NumPy 卷积到 PyTorch 识别 MNIST 手写数字

你先用 NumPy 手写单通道卷积和池化，用固定的小矩阵验证它，再和 `nn.Conv2d` 逐项对照，然后用 PyTorch 搭建、训练、保存并测试一个 CNN 分类器。核心函数保留了完整签名、docstring 和 TODO 提示，函数体留给你实现。公式、API 链接、自检单元格和全部绘图代码已经提供。

## 使用方法

1. 在本目录运行 `uv sync`，然后在 VS Code / Jupyter 中选择 `.venv/bin/python` 作为 kernel。
2. 从上往下运行。遇到 `TODO` 函数时先实现它，再运行紧随其后的自检单元格。
3. 修改函数后要重新运行定义它的单元格，编辑器里的代码不会自动同步到 kernel。
4. 没实现的函数会抛出 `NotImplementedError`，这是练习刚开始时的正常状态。
5. 保持函数名、参数顺序、返回值和 `nn.Module` 里的属性名不变（`conv1`、`pool1`、`conv2`、`pool2`、`fc1`、`fc2`），后面的自检和绘图依赖它们。
6. 全部完成后执行 Restart Kernel → Run All，确认结果不依赖旧变量。

**规则**：Lab 1 只能用 NumPy 实现卷积和池化，不要调用 `torch` 或 `scipy`，PyTorch 只出现在自检里当参照答案。Lab 2 起可以使用 PyTorch。

## 路线

| Lab | 内容 | 你要实现的函数 | 已提供的观察图 |
| --- | --- | --- | --- |
| 1 | NumPy 卷积 / 池化 | `conv_output_size`, `conv2d_forward`, `maxpool2d_forward`, `conv2d_multi_channel`（进阶） | 滑动窗口逐步演示、池化演示 |
| 2 | 对照 `nn.Conv2d` | `torch_conv_from_numpy`, `conv_param_count`, `receptive_field` | 局部连接 / 参数共享的参数量对比、权重梯度的累加 |
| 3 | 搭建 CNN | `SimpleCNN.__init__`, `SimpleCNN.forward` | 每层输出 shape 表 |
| 4 | 数据与训练流程 | `make_dataloaders`, `train_one_epoch`, `evaluate`, `save_checkpoint`, `load_checkpoint`, `fit` | 先过拟合 32 张图，再训练全量数据，画 loss / accuracy 曲线 |
| 5 | 结果分析 | `get_first_layer_feature_maps`, `get_misclassified` | 第一层卷积核与 feature maps、分类错误图片、混淆矩阵 |
| 6 | 对比与研究 | 无 | NumPy 与 PyTorch 的速度对比，进阶 TODO 清单 |

一共 17 个函数 / 方法（含 1 个进阶函数；`SimpleCNN` 的 `__init__` 和 `forward` 分开计数）。

## 1. Problem Definition

**任务**：输入一张 $28\times28$ 的灰度图片，预测它是 0 到 9 中的哪个数字。

| 项目 | 本实验的约定 |
| --- | --- |
| 输入 | `[B, 1, 28, 28]`（`float32`，已按均值 0.1307 / 标准差 0.3081 标准化） |
| 标签 | `[B]`，整数类别 0–9（`int64`） |
| 输出 | `[B, 10]`，10 个 **logits**（不要在模型里加 Softmax） |
| 数据划分 | 官方训练集 60,000 张拆成 55,000 训练 + 5,000 验证；官方测试集 10,000 张只在最后评估一次 |
| 损失 | `nn.CrossEntropyLoss`（内部包含 log-softmax，直接接收 logits） |
| 优化器 | Adam，学习率 0.001 |
| Batch size | 64 |
| Epochs | 先训练 5 轮 |

**目标**：测试准确率 98% 左右。这个数字不是重点，重点是你要在每一步确认维度、梯度和训练流程是对的。

**验收顺序**：先让模型把固定的 32 张训练图记到 100% 准确率（Lab 4），这说明网络、loss、梯度、优化器接得没问题。然后再训练全量数据。

## 2. 环境与 API 速查

下表只列出实现所需的**部分参数**，不是 API 的全部参数。写 PyTorch 代码时以官方文档为准，版本不同参数可能有差别。

### 2.1 NumPy（Lab 1）

| API（官方文档） | 关键参数 / 返回值 | 用在哪里 |
| --- | --- | --- |
| [`np.pad(array, pad_width, mode="constant")`](https://numpy.org/doc/stable/reference/generated/numpy.pad.html) | `pad_width` 给整数时每个轴两侧都补这么多；默认补 0 | 卷积的 zero padding |
| [`np.zeros(shape)`](https://numpy.org/doc/stable/reference/generated/numpy.zeros.html) | `shape` 是元组 | 创建输出数组 |
| [数组切片 `a[i:i+k, j:j+k]`](https://numpy.org/doc/stable/user/basics.indexing.html) | 取出一个窗口，**是视图不是拷贝** | 取当前卷积窗口 |
| [`np.sum(a)`](https://numpy.org/doc/stable/reference/generated/numpy.sum.html)、[`np.max(a)`](https://numpy.org/doc/stable/reference/generated/numpy.max.html) | 对窗口求和 / 求最大值 | 卷积、最大池化 |
| [`np.stack(arrays, axis=0)`](https://numpy.org/doc/stable/reference/generated/numpy.stack.html) | 沿新轴堆叠 | 多通道输出 |

### 2.2 PyTorch 模块（Lab 2、3）

| API（官方文档） | 关键参数 / 返回值 | 用在哪里 |
| --- | --- | --- |
| [`nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0)`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) | 输入 `[B, C_in, H, W]`，输出 `[B, C_out, H_out, W_out]`；权重 `weight` 形状 `[C_out, C_in, kH, kW]`，`bias` 形状 `[C_out]` | 卷积层 |
| [`nn.MaxPool2d(kernel_size, stride=None)`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) | `stride` 默认等于 `kernel_size` | 池化层 |
| [`nn.Linear(in_features, out_features)`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html) | `weight` 形状 `[out, in]` | 全连接层 |
| [`F.relu(x)`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.relu.html) | 逐元素 $\max(0,x)$ | 激活 |
| [`Tensor.flatten(start_dim=1)`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.flatten.html) | 从 `start_dim` 起展平，保留 batch 维 | 卷积特征接全连接层 |
| [`nn.Module`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html) | 在 `__init__` 里把层赋给 `self.xxx`，参数才会被注册；`forward` 里写前向 | 自定义网络 |
| [`torch.no_grad()`](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html) | 上下文内不记录计算图 | 评估、改权重、画 feature maps |

### 2.3 数据与训练流程（Lab 4）

| API（官方文档） | 关键参数 / 返回值 | 用在哪里 |
| --- | --- | --- |
| [`torchvision.datasets.MNIST(root, train, download, transform)`](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.MNIST.html) | `train=True` 是 60,000 张训练集，`False` 是 10,000 张测试集；每个样本是 `(image, label)` | 加载数据 |
| [`transforms.Compose`](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.Compose.html)、[`ToTensor`](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.ToTensor.html)、[`Normalize(mean, std)`](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.Normalize.html) | `ToTensor` 把 PIL 图变成 `[1, 28, 28]` 且数值缩放到 $[0,1]$；`Normalize` 做 $(x-\text{mean})/\text{std}$ | 预处理 |
| [`random_split(dataset, lengths, generator)`](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.random_split) | 传入带种子的 `torch.Generator` 才可复现 | 55,000 / 5,000 划分 |
| [`DataLoader(dataset, batch_size, shuffle)`](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) | 训练集 `shuffle=True`，验证 / 测试集 `False` | 小批量迭代 |
| [`nn.CrossEntropyLoss()`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) | 输入 logits `[B, C]`，目标是类别下标 `[B]`（`int64`），默认返回 batch 平均 | 损失 |
| [`torch.optim.Adam(params, lr)`](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html) | `params` 通常是 `model.parameters()` | 优化器 |
| [`optimizer.zero_grad()` / `loss.backward()` / `optimizer.step()`](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html) | 每个 batch 的固定三步，顺序不能乱 | 训练循环 |
| [`model.train()` / `model.eval()`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.train) | 切换 Dropout / BatchNorm 的行为 | 训练与评估 |
| [`Tensor.argmax(dim)`](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html) | 沿 `dim` 取最大值的下标 | 从 logits 得到预测类别 |
| [`torch.save(obj, path)`](https://docs.pytorch.org/docs/stable/generated/torch.save.html) / [`torch.load(path, map_location, weights_only=True)`](https://docs.pytorch.org/docs/stable/generated/torch.load.html) | 保存 / 读取 `model.state_dict()` | 保存和恢复模型 |
| [`model.load_state_dict(state)`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.load_state_dict) | 要求键名和形状一致 | 恢复权重 |

官方参考：[PyTorch 图像分类教程](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)、[Quickstart](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html)、[`nn.Conv2d` 文档](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)。

> **TODO（研究）**：打开 `nn.Conv2d` 文档的 *Shape* 和 *Variables* 两节，确认 `weight` 的形状、`bias` 的形状和输出尺寸公式，对照第 4 节的公式。再看一下文档里 `padding="same"` 的说明，它在什么条件下可用？

### 2.4 NumPy 手写、PyTorch、TensorFlow / Keras 对比

本实验主线用 PyTorch。TensorFlow / Keras 在这里只做对照，不安装、不运行。

| 维度 | NumPy 手写（Lab 1） | PyTorch（本实验主线） | TensorFlow / Keras |
| --- | --- | --- | --- |
| 卷积实现 | 4 层 Python 循环，只能用来理解 | 底层 C++/CUDA/MPS kernel（cuDNN 等） | 同样调用优化后的 kernel |
| 反向传播 | 要自己推导和写 | autograd 自动完成 | `tf.GradientTape` 或 `model.fit` 自动完成 |
| 张量布局 | 自定，本实验用 `[C, H, W]` | `[B, C, H, W]`（channels-first） | 默认 `[B, H, W, C]`（channels-last） |
| 卷积核形状 | 自定 | `[C_out, C_in, kH, kW]` | `[kH, kW, C_in, C_out]` |
| 训练循环 | 全部手写 | 显式写 `zero_grad → backward → step` | `model.compile` + `model.fit` 封装，也可自定义 |
| 调试体验 | 最透明 | 动态图，可以直接 `print` 和断点 | Eager 模式类似，`@tf.function` 里会变成图 |
| 速度（本实验规模） | 慢几个数量级（Lab 6 实测） | 快 | 与 PyTorch 同量级 |
| 适合学什么 | 弄懂卷积到底在算什么 | 搞清训练流程的每一步 | 了解高层封装怎么省代码 |

**怎么选**：NumPy 版是为了让你看见卷积的每个乘加；PyTorch 版是为了让你自己控制训练的每一步。Keras 的 `fit` 会把这些步骤藏起来，等你清楚每一步之后再用也不迟。

> **TODO（研究）**：用 Keras 写出同一个网络（`Conv2D(16, 3, padding="same")` 开始），逐层对照输出 shape 和参数量（Keras `model.summary()`）。注意 `Conv2D` 的卷积核布局和 PyTorch 不同，权重互相迁移时要转置。

In [ ]:
import time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("Matplotlib:", matplotlib.__version__)

# 统一的绘图样式：两个系列用蓝 / 橙区分，并配合线型作为第二重编码。
SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7"]
# 单一色相，从浅到深，用于 feature map 和混淆矩阵。
SEQ_CMAP = LinearSegmentedColormap.from_list("seq", ["#fcfcfb", "#2a78d6", "#0d3a73"])
HILITE = "#eb6834"  # 高亮色：当前窗口、错误预测

# 中文字体回退：依次尝试 macOS / Windows / Linux 上常见的中文字体。
plt.rcParams["font.sans-serif"] = [
    "PingFang SC", "Heiti SC", "Arial Unicode MS", "Microsoft YaHei", "SimHei",
    "Noto Sans CJK SC", "WenQuanYi Micro Hei", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 11,
})


def get_device() -> torch.device:
    """优先 CUDA，其次 Apple MPS，最后 CPU。"""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = get_device()
print("Device:", DEVICE)

## 3. 核心参数配置

下面是第一版固定使用的配置，它们是实验起点，不是最优配置。

| Hyperparameter | 代码常量 | 值 | 说明 |
| --- | --- | --- | --- |
| 数据目录 | `DATA_DIR` | `"data"` | 首次运行会下载约 11 MB，已被本目录 `.gitignore` 排除 |
| 训练 / 验证张数 | `N_TRAIN`, `N_VAL` | 55,000 / 5,000 | 官方训练集的拆分 |
| Batch size | `BATCH_SIZE` | 64 | |
| Learning rate | `LEARNING_RATE` | 0.001 | Adam |
| Epochs | `EPOCHS` | 5 | |
| Random seed | `SEED` | 42 | 划分数据和初始化权重都用它 |
| 标准化均值 / 标准差 | `MNIST_MEAN`, `MNIST_STD` | 0.1307 / 0.3081 | MNIST 训练集像素的均值和标准差 |
| 检查点路径 | `CHECKPOINT_PATH` | `"mnist_cnn_best.pt"` | 验证集准确率最高时保存 |

In [ ]:
DATA_DIR = "data"
N_TRAIN, N_VAL = 55_000, 5_000
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
EPOCHS = 5
SEED = 42
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081
CHECKPOINT_PATH = "mnist_cnn_best.pt"
NUM_CLASSES = 10

torch.manual_seed(SEED)
np.random.seed(SEED)

## 4. 公式总表

实现时随时回来查这一节。每个 Lab 开头还会重复它用到的公式。

### 4.1 卷积（实际是 cross-correlation，不翻转卷积核）

单通道，输入 $X\in\mathbb R^{H\times W}$，卷积核 $K\in\mathbb R^{k_h\times k_w}$，偏置 $b$，输出位置 $(i,j)$：

$$
Y_{i,j}=b+\sum_{u=0}^{k_h-1}\sum_{v=0}^{k_w-1}K_{u,v}\;X^{\text{pad}}_{\,i\cdot s+u,\;j\cdot s+v}
$$

$X^{\text{pad}}$ 是在 $X$ 四周补了 $p$ 圈 0 的矩阵，$s$ 是 stride。

**输出尺寸**（每个空间维度单独算，向下取整）：

$$
H_{\text{out}}=\left\lfloor\frac{H+2p-k_h}{s}\right\rfloor+1,\qquad
W_{\text{out}}=\left\lfloor\frac{W+2p-k_w}{s}\right\rfloor+1
$$

**多通道**：输入 $C_{\text{in}}$ 个通道，输出 $C_{\text{out}}$ 个通道。第 $o$ 个输出通道把所有输入通道的卷积结果**相加**，再加这个输出通道自己的 bias：

$$
Y^{(o)}=b^{(o)}+\sum_{c=0}^{C_{\text{in}}-1}K^{(o,c)}\star X^{(c)}
$$

权重形状 `[C_out, C_in, kH, kW]`，bias 形状 `[C_out]`。

### 4.2 参数量

$$
\#\text{params}=C_{\text{out}}\bigl(C_{\text{in}}\,k_h\,k_w+1\bigr)
$$

其中 $+1$ 是每个输出通道一个 bias。全连接层是 $n_{\text{out}}(n_{\text{in}}+1)$。

### 4.3 最大池化

$$
Y_{i,j}=\max_{0\le u<k,\;0\le v<k}X_{\,i\cdot s+u,\;j\cdot s+v}
$$

输出尺寸公式同 4.1（$p=0$）。池化**没有可学习参数**，按通道独立计算。

### 4.4 感受野（receptive field）

从输入端往后逐层递推。记 $r$ 为感受野边长，$j$ 为相邻两个输出在输入上的间距（jump），初始 $r_0=1,\ j_0=1$。第 $l$ 层的核大小是 $k_l$、步长是 $s_l$：

$$
r_l=r_{l-1}+(k_l-1)\,j_{l-1},\qquad j_l=j_{l-1}\,s_l
$$

padding 不改变感受野大小。

### 4.5 Softmax 与 Cross-Entropy

模型输出 logits $z\in\mathbb R^{10}$，真实类别 $y$：

$$
p_c=\frac{e^{z_c}}{\sum_{c'}e^{z_{c'}}},\qquad
\mathcal L=-\frac1B\sum_{n=1}^{B}\log p^{(n)}_{y^{(n)}}
=\frac1B\sum_{n=1}^B\Bigl(\log\sum_{c'}e^{z^{(n)}_{c'}}-z^{(n)}_{y^{(n)}}\Bigr)
$$

`nn.CrossEntropyLoss` 直接做第二个等号右边的稳定版本，所以**模型最后不要再加 Softmax**。预测类别用 $\hat y=\arg\max_c z_c$，不需要先算 Softmax。

### 4.6 本实验网络的形状与参数量

| 层 | 配置 | 输出 shape | 参数量 |
| --- | --- | --- | --- |
| 输入 | 灰度图 | `[B, 1, 28, 28]` | 0 |
| Conv + ReLU | 1→16, k=3, p=1 | `[B, 16, 28, 28]` | $16(1\cdot9+1)=160$ |
| MaxPool | k=2, s=2 | `[B, 16, 14, 14]` | 0 |
| Conv + ReLU | 16→32, k=3, p=1 | `[B, 32, 14, 14]` | $32(16\cdot9+1)=4640$ |
| MaxPool | k=2, s=2 | `[B, 32, 7, 7]` | 0 |
| Flatten | | `[B, 1568]` | 0 |
| Linear + ReLU | 1568→128 | `[B, 128]` | $128(1568+1)=200832$ |
| Linear | 128→10 | `[B, 10]` | $10(128+1)=1290$ |

合计 $160+4640+200832+1290=206922$ 个可训练参数。大部分参数在第一个全连接层里。

## Lab 1：用 NumPy 手写卷积和池化

先写单通道版本。写的时候请分三步走，每一步通过后再加下一个功能：

1. 只支持 `stride=1, padding=0`，用 4.1 的公式对着写。
2. 加上 `padding`（先用 `np.pad` 补 0，再按原来的方式滑窗）。
3. 加上 `stride`（窗口左上角每次移动 `stride` 格）。

**你要实现**：

- `conv_output_size(n_in, kernel, stride=1, padding=0)`：4.1 的输出尺寸公式，整数除法用 `//`。
- `conv2d_forward(x, w, b=0.0, stride=1, padding=0)`：`x` 是 `(H, W)`，`w` 是 `(kH, kW)`，返回 `(H_out, W_out)`。
- `maxpool2d_forward(x, kernel=2, stride=2)`：单通道最大池化。
- `conv2d_multi_channel(x, w, b, stride=1, padding=0)`（进阶）：`x` 是 `(C_in, H, W)`，`w` 是 `(C_out, C_in, kH, kW)`，`b` 是 `(C_out,)`，返回 `(C_out, H_out, W_out)`。请复用 `conv2d_forward`。

**提示**：

- 卷积核**不翻转**（深度学习里的“卷积”就是 cross-correlation），和 PyTorch 一致。
- 用 `x_pad[i*stride : i*stride+kh, j*stride : j*stride+kw]` 取窗口，再和 `w` 逐元素相乘求和。
- 不要改动输入数组 `x`。`np.pad` 返回新数组，不会修改原数组。
- 取窗口取的是 `x_pad`，不是 `x`。

In [ ]:
def conv_output_size(n_in: int, kernel: int, stride: int = 1, padding: int = 0) -> int:
    """卷积 / 池化在一个空间维度上的输出长度。

    Parameters
    ----------
    n_in : int
        输入长度（H 或 W）。
    kernel : int
        卷积核（或池化窗口）边长。
    stride : int
        步长，必须 >= 1。
    padding : int
        每一侧补 0 的数量。

    Returns
    -------
    int
        floor((n_in + 2 * padding - kernel) / stride) + 1
    """
    # TODO: 按 4.1 的公式返回整数。
    raise NotImplementedError("TODO Lab 1: implement conv_output_size")

In [ ]:
def conv2d_forward(
    x: np.ndarray,
    w: np.ndarray,
    b: float = 0.0,
    stride: int = 1,
    padding: int = 0,
) -> np.ndarray:
    """单通道 2D 卷积（cross-correlation）的前向传播。

    Parameters
    ----------
    x : ndarray, shape (H, W)
        输入图片。
    w : ndarray, shape (kH, kW)
        卷积核。
    b : float
        偏置，加到每个输出位置上。
    stride : int
        窗口每次移动的格数。
    padding : int
        四周各补 padding 圈 0。

    Returns
    -------
    out : ndarray, shape (H_out, W_out)

    Notes
    -----
    不修改 x。卷积核不翻转。
    """
    # TODO 步骤 1：只支持 stride=1, padding=0。
    #   kh, kw = w.shape；用 conv_output_size 算出 H_out, W_out；
    #   out = np.zeros((H_out, W_out))；双层循环取窗口，np.sum(窗口 * w) + b。
    # TODO 步骤 2：先用 np.pad 把 x 补成 x_pad，后面都从 x_pad 里取窗口。
    # TODO 步骤 3：窗口左上角改成 (i * stride, j * stride)。
    raise NotImplementedError("TODO Lab 1: implement conv2d_forward")

### Lab 1 自检

先用一个可以手算的 $4\times4$ 输入和竖直边缘卷积核：

$$
X=\begin{bmatrix}1&2&3&0\\0&1&2&3\\3&0&1&2\\2&3&0&1\end{bmatrix},\qquad
K=\begin{bmatrix}1&0&-1\\1&0&-1\\1&0&-1\end{bmatrix}
$$

左上角窗口是 $\begin{bmatrix}1&2&3\\0&1&2\\3&0&1\end{bmatrix}$，左列之和 $1+0+3=4$，右列之和 $3+2+1=6$，所以 $Y_{0,0}=4-6=-2$。先自己把另外三个位置算出来，再运行自检看你算得对不对。

In [ ]:
# Lab 1 自检 A：stride=1, padding=0（手算结果）
X_hand = np.array([[1, 2, 3, 0], [0, 1, 2, 3], [3, 0, 1, 2], [2, 3, 0, 1]], dtype=float)
K_edge = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]], dtype=float)
X_before = X_hand.copy()

for n, k, s, p, expect in [(4, 3, 1, 0, 2), (4, 3, 1, 1, 4), (4, 3, 2, 1, 2), (28, 3, 1, 1, 28), (28, 2, 2, 0, 14), (7, 3, 2, 0, 3)]:
    assert conv_output_size(n, k, s, p) == expect, f"conv_output_size({n}, {k}, {s}, {p}) 应为 {expect}"

out_a = conv2d_forward(X_hand, K_edge)
assert out_a.shape == (2, 2), f"形状应为 (2, 2)，得到 {out_a.shape}"
np.testing.assert_allclose(out_a, [[-2, -2], [2, -2]])
np.testing.assert_allclose(conv2d_forward(X_hand, K_edge, b=1.5), [[-0.5, -0.5], [3.5, -0.5]])
np.testing.assert_array_equal(X_hand, X_before)
print("Lab 1 自检 A 通过：手算结果一致。")

In [ ]:
# Lab 1 自检 B：加入 padding。角落窗口里有补出来的 0。
out_b = conv2d_forward(X_hand, K_edge, padding=1)
assert out_b.shape == (4, 4), f"形状应为 (4, 4)，得到 {out_b.shape}"
# 左上角窗口 [[0,0,0],[0,1,2],[0,0,1]]：左列和 0，右列和 0+2+1=3，所以是 -3。
assert out_b[0, 0] == -3.0, f"out[0,0] 应为 -3，得到 {out_b[0, 0]}"
# 与 PyTorch 对照（只在自检里用）
ref_b = F.conv2d(torch.tensor(X_hand)[None, None], torch.tensor(K_edge)[None, None], padding=1)[0, 0].numpy()
np.testing.assert_allclose(out_b, ref_b)
print("Lab 1 自检 B 通过：padding 正确。")

In [ ]:
# Lab 1 自检 C：加入 stride，并在随机输入上扫描多种 (stride, padding) 组合。
rng = np.random.default_rng(0)
for h, w_, kh, kw in [(8, 8, 3, 3), (9, 7, 3, 3), (10, 10, 5, 5), (6, 8, 2, 4)]:
    x = rng.standard_normal((h, w_)); k = rng.standard_normal((kh, kw)); bias = float(rng.standard_normal())
    for stride in (1, 2, 3):
        for padding in (0, 1, 2):
            mine = conv2d_forward(x, k, bias, stride=stride, padding=padding)
            ref = F.conv2d(
                torch.tensor(x)[None, None], torch.tensor(k)[None, None],
                torch.tensor([bias], dtype=torch.float64), stride=stride, padding=padding,
            )[0, 0].numpy()
            assert mine.shape == ref.shape, f"stride={stride}, padding={padding} 形状 {mine.shape} != {ref.shape}"
            np.testing.assert_allclose(mine, ref, atol=1e-10, err_msg=f"stride={stride}, padding={padding}")
            assert conv_output_size(h, kh, stride, padding) == ref.shape[0]
            assert conv_output_size(w_, kw, stride, padding) == ref.shape[1]
print("Lab 1 自检 C 通过：36 种配置都和 F.conv2d 一致。")

### 观察：卷积窗口一步一步滑过输入

下面的绘图代码已经提供。每一行显示一个输出位置：左边是输入（橙色框是当前窗口，虚线是 padding 补出来的 0），中间是卷积核，右边是输出（橙色格子是刚算出来的这个位置）。把 `STRIDE` 和 `PADDING` 改成你想看的值，对照窗口位置和输出尺寸。

In [ ]:
def draw_conv_steps(x, w, positions, stride=1, padding=0, b=0.0):
    """画出若干个输出位置的卷积过程。依赖你实现的 conv2d_forward。"""
    out = conv2d_forward(x, w, b, stride=stride, padding=padding)
    x_pad = np.pad(x, padding)
    kh, kw = w.shape
    fig, axes = plt.subplots(len(positions), 3, figsize=(9.5, 3.0 * len(positions)))
    axes = np.atleast_2d(axes)
    for row, (i, j) in zip(axes, positions):
        ax_x, ax_k, ax_o = row
        ax_x.imshow(x_pad, cmap=SEQ_CMAP, vmin=x_pad.min(), vmax=max(x_pad.max(), 1))
        ax_x.add_patch(Rectangle((j * stride - 0.5, i * stride - 0.5), kw, kh, fill=False, ec=HILITE, lw=2.5))
        if padding:
            ax_x.add_patch(Rectangle((padding - 0.5, padding - 0.5), x.shape[1], x.shape[0], fill=False, ec="#555", lw=1, ls="--"))
        for (r, c), v in np.ndenumerate(x_pad):
            ax_x.text(c, r, f"{v:g}", ha="center", va="center", fontsize=8, color="#222" if v < 0.6 * max(x_pad.max(), 1) else "white")
        ax_x.set_title(f"输入（含 padding={padding}）  窗口左上角 ({i * stride}, {j * stride})")
        ax_k.imshow(w, cmap="RdBu", vmin=-abs(w).max(), vmax=abs(w).max())
        for (r, c), v in np.ndenumerate(w):
            ax_k.text(c, r, f"{v:g}", ha="center", va="center", fontsize=10)
        window = x_pad[i * stride : i * stride + kh, j * stride : j * stride + kw]
        ax_k.set_title("卷积核")
        ax_o.imshow(out, cmap=SEQ_CMAP)
        ax_o.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec=HILITE, lw=2.5))
        for (r, c), v in np.ndenumerate(out):
            ax_o.text(c, r, f"{v:g}", ha="center", va="center", fontsize=9, color="#222" if v < out.max() * 0.6 else "white")
        ax_o.set_title(f"输出 [{i},{j}] = Σ(窗口×核)+b = {np.sum(window * w) + b:g}")
        for a in row:
            a.set_xticks([]); a.set_yticks([])
    fig.tight_layout()
    plt.show()


STRIDE, PADDING = 1, 0
draw_conv_steps(X_hand, K_edge, [(0, 0), (0, 1), (1, 1)], stride=STRIDE, padding=PADDING)

### 最大池化

**你要实现**：`maxpool2d_forward(x, kernel=2, stride=2)`，公式见 4.3。取窗口的方式和卷积一样，只是把“乘加”换成 `np.max`。池化没有权重，也没有 padding。

In [ ]:
def maxpool2d_forward(x: np.ndarray, kernel: int = 2, stride: int = 2) -> np.ndarray:
    """单通道最大池化。

    Parameters
    ----------
    x : ndarray, shape (H, W)
    kernel : int
        正方形池化窗口的边长。
    stride : int
        窗口移动步长。

    Returns
    -------
    out : ndarray, shape (H_out, W_out)
        每个窗口里的最大值。
    """
    # TODO: 用 conv_output_size(…, padding=0) 算输出尺寸，双层循环取窗口，np.max 取最大值。
    raise NotImplementedError("TODO Lab 1: implement maxpool2d_forward")

In [ ]:
# Lab 1 自检：最大池化
P = np.array([[1, 3, 2, 0], [4, 2, 1, 5], [0, 1, 7, 2], [3, 2, 1, 6]], dtype=float)
np.testing.assert_allclose(maxpool2d_forward(P), [[4, 5], [3, 7]])
# 重叠窗口：kernel=3, stride=1 -> 输出 (2, 2)，四个窗口都包含中间的 7
np.testing.assert_allclose(maxpool2d_forward(P, kernel=3, stride=1), [[7, 7], [7, 7]])
# 奇数尺寸会丢掉边缘（向下取整）：5x5, k=2, s=2 -> 2x2
assert maxpool2d_forward(np.arange(25.0).reshape(5, 5)).shape == (2, 2)
# 与 PyTorch 对照
r = rng.standard_normal((12, 12))
ref = F.max_pool2d(torch.tensor(r)[None, None], kernel_size=2, stride=2)[0, 0].numpy()
np.testing.assert_allclose(maxpool2d_forward(r), ref)
# 同一个池化窗口内的位置变化会被吸收：亮点从 (0,0) 挪到 (1,1)，两者在同一个 2x2 窗口里，池化输出相同
a = np.zeros((4, 4)); a[0, 0] = 1
bb = np.zeros((4, 4)); bb[1, 1] = 1
np.testing.assert_allclose(maxpool2d_forward(a), maxpool2d_forward(bb))
print("Lab 1 maxpool2d_forward 自检通过。")

### 进阶：多通道卷积

**你要实现**：`conv2d_multi_channel(x, w, b, stride=1, padding=0)`，公式见 4.1 的“多通道”。对每个输出通道 $o$，把所有输入通道的单通道卷积结果相加，再加上 $b^{(o)}$。

注意：`conv2d_forward` 里已经会加一个 `b`，多通道时只能让它加**一次**。简单做法是调用 `conv2d_forward` 时把 `b` 留成 0，最后再统一加 `b[o]`。

In [ ]:
def conv2d_multi_channel(
    x: np.ndarray,
    w: np.ndarray,
    b: np.ndarray,
    stride: int = 1,
    padding: int = 0,
) -> np.ndarray:
    """多通道卷积的前向传播（单个样本，没有 batch 维）。

    Parameters
    ----------
    x : ndarray, shape (C_in, H, W)
    w : ndarray, shape (C_out, C_in, kH, kW)
    b : ndarray, shape (C_out,)
    stride, padding : int

    Returns
    -------
    out : ndarray, shape (C_out, H_out, W_out)
    """
    # TODO: 对每个输出通道 o：
    #   acc = 所有输入通道 c 的 conv2d_forward(x[c], w[o, c], 0.0, stride, padding) 之和；
    #   然后加 b[o]。最后用 np.stack 把 C_out 个结果叠成 (C_out, H_out, W_out)。
    raise NotImplementedError("TODO Lab 1: implement conv2d_multi_channel")

In [ ]:
# Lab 1 进阶自检：多通道卷积与 F.conv2d 对照
x = rng.standard_normal((3, 9, 9)); w = rng.standard_normal((4, 3, 3, 3)); bias = rng.standard_normal(4)
for stride, padding in [(1, 0), (1, 1), (2, 1)]:
    mine = conv2d_multi_channel(x, w, bias, stride, padding)
    ref = F.conv2d(torch.tensor(x)[None], torch.tensor(w), torch.tensor(bias), stride=stride, padding=padding)[0].numpy()
    assert mine.shape == ref.shape, f"形状 {mine.shape} != {ref.shape}"
    np.testing.assert_allclose(mine, ref, atol=1e-10)
print("Lab 1 进阶 conv2d_multi_channel 自检通过。")

### 观察：最大池化

下面的绘图代码已经提供：左边是输入，右边是池化结果，橙色框是被选中的最大值。

In [ ]:
def draw_maxpool(x, kernel=2, stride=2):
    out = maxpool2d_forward(x, kernel, stride)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(8, 3.8))
    a1.imshow(x, cmap=SEQ_CMAP)
    for (r, c), v in np.ndenumerate(x):
        a1.text(c, r, f"{v:g}", ha="center", va="center", color="#222" if v < 0.6 * x.max() else "white")
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            win = x[i * stride : i * stride + kernel, j * stride : j * stride + kernel]
            a1.add_patch(Rectangle((j * stride - 0.5, i * stride - 0.5), kernel, kernel, fill=False, ec="#999", lw=1, ls="--"))
            rr, cc = np.unravel_index(np.argmax(win), win.shape)
            a1.add_patch(Rectangle((j * stride + cc - 0.5, i * stride + rr - 0.5), 1, 1, fill=False, ec=HILITE, lw=2.5))
    a1.set_title(f"输入 {x.shape}（虚线 = 窗口，橙框 = 最大值）")
    a2.imshow(out, cmap=SEQ_CMAP)
    for (r, c), v in np.ndenumerate(out):
        a2.text(c, r, f"{v:g}", ha="center", va="center", color="#222" if v < 0.6 * out.max() else "white")
    a2.set_title(f"输出 {out.shape}")
    for a in (a1, a2):
        a.set_xticks([]); a.set_yticks([])
    fig.tight_layout(); plt.show()


draw_maxpool(P)

> **TODO（研究）**：
> 1. 你的 `conv2d_forward` 有 4 层循环（输出行、输出列，加上 `np.sum` 里隐含的核行、核列）。查一下 **im2col**：它怎样把卷积变成一次矩阵乘法？为什么 GPU 上的卷积实现常常基于它？可以用 `np.lib.stride_tricks.sliding_window_view` 先试着把 `conv2d_forward` 改成不用 Python 循环的版本，用 Lab 6 的计时对比。
> 2. 手写卷积的**反向传播**：已知 $\partial\mathcal L/\partial Y$，推导 $\partial\mathcal L/\partial K$ 和 $\partial\mathcal L/\partial X$。提示：$\partial\mathcal L/\partial K_{u,v}=\sum_{i,j}\frac{\partial\mathcal L}{\partial Y_{i,j}}X^{\text{pad}}_{i s+u,\,j s+v}$，$\partial\mathcal L/\partial X$ 对应一次“转置卷积”（full convolution）。Lab 2 会让你看到权重梯度怎样从所有输出位置累加。

## Lab 2：对照 `nn.Conv2d`，并弄清几个概念

**你要实现**：

- `torch_conv_from_numpy(w, b, stride=1, padding=0)`：创建一个 `nn.Conv2d(1, 1, kernel_size=w.shape, ...)`，把 NumPy 的 `w`、`b` 复制进去，返回这个层。
- `conv_param_count(c_in, c_out, kh, kw)`：4.2 的参数量公式。
- `receptive_field(layers)`：4.4 的感受野递推，`layers` 是 `[(kernel, stride), ...]`。

**提示**：

- 改权重要放进 `with torch.no_grad():`，并用 `conv.weight.copy_(...)`、`conv.bias.copy_(...)`。`conv.weight` 的形状是 `[1, 1, kH, kW]`，`conv.bias` 的形状是 `[1]`。
- NumPy 数组转 Tensor：`torch.from_numpy(w).float()`（`nn.Conv2d` 默认权重是 `float32`）。
- 只用 `padding` 参数的整数形式。

In [ ]:
def torch_conv_from_numpy(w: np.ndarray, b: float, stride: int = 1, padding: int = 0) -> nn.Conv2d:
    """创建单通道 nn.Conv2d，并把 NumPy 的卷积核和偏置复制进去。

    Parameters
    ----------
    w : ndarray, shape (kH, kW)
    b : float
    stride, padding : int

    Returns
    -------
    conv : nn.Conv2d
        in_channels=1, out_channels=1，权重和偏置等于 w、b。
    """
    # TODO: conv = nn.Conv2d(1, 1, kernel_size=w.shape, stride=stride, padding=padding)
    #       在 torch.no_grad() 里，用 copy_ 把 w 放进 conv.weight[0, 0]，b 放进 conv.bias。
    raise NotImplementedError("TODO Lab 2: implement torch_conv_from_numpy")

In [ ]:
def conv_param_count(c_in: int, c_out: int, kh: int, kw: int) -> int:
    """普通卷积层的可训练参数量：C_out * (C_in * kH * kW + 1)。"""
    # TODO: 按 4.2 的公式返回整数。
    raise NotImplementedError("TODO Lab 2: implement conv_param_count")

In [ ]:
def receptive_field(layers: list[tuple[int, int]]) -> int:
    """按 4.4 的递推计算最后一层输出单元的感受野边长。

    Parameters
    ----------
    layers : list of (kernel, stride)
        从输入端到输出端排列，卷积和池化都按 (核边长, 步长) 给出。

    Returns
    -------
    int
        感受野边长。
    """
    # TODO: r = 1, j = 1；对每个 (k, s)：r += (k - 1) * j；j *= s。返回 r。
    raise NotImplementedError("TODO Lab 2: implement receptive_field")

In [ ]:
# Lab 2 自检 A：同一组权重，NumPy 手写版和 nn.Conv2d 的输出一致
conv_t = torch_conv_from_numpy(K_edge, 0.5, stride=1, padding=0)
assert isinstance(conv_t, nn.Conv2d)
assert conv_t.weight.shape == (1, 1, 3, 3) and conv_t.bias.shape == (1,)
out_t = conv_t(torch.tensor(X_hand, dtype=torch.float32)[None, None])
assert out_t.shape == (1, 1, 2, 2), "输出应为 [B, C_out, H_out, W_out]"
np.testing.assert_allclose(out_t[0, 0].detach().numpy(), conv2d_forward(X_hand, K_edge, 0.5), atol=1e-5)
np.testing.assert_allclose(out_t[0, 0].detach().numpy(), [[-1.5, -1.5], [2.5, -1.5]], atol=1e-5)

# 随机输入、带 stride 和 padding
x = rng.standard_normal((10, 10)); k = rng.standard_normal((3, 3)); bias = 0.3
for s, p in [(1, 0), (1, 1), (2, 1), (3, 2)]:
    c = torch_conv_from_numpy(k, bias, s, p)
    y_t = c(torch.tensor(x, dtype=torch.float32)[None, None])[0, 0].detach().numpy()
    np.testing.assert_allclose(y_t, conv2d_forward(x, k, bias, s, p), atol=1e-5, err_msg=f"s={s}, p={p}")
print("Lab 2 自检 A 通过：nn.Conv2d 与手写版一致。")

# Lab 2 自检 B：参数量与感受野
assert conv_param_count(1, 16, 3, 3) == 160
assert conv_param_count(16, 32, 3, 3) == 4640
assert conv_param_count(3, 64, 7, 7) == 9472
assert conv_param_count(1, 1, 3, 3) == sum(p.numel() for p in nn.Conv2d(1, 1, 3).parameters())
assert conv_param_count(16, 32, 3, 3) == sum(p.numel() for p in nn.Conv2d(16, 32, 3).parameters())
assert receptive_field([(3, 1)]) == 3
assert receptive_field([(3, 1), (3, 1)]) == 5          # 两个 3x3 叠在一起等于一个 5x5
assert receptive_field([(3, 1), (2, 2), (3, 1)]) == 8  # 池化让后面的卷积“看得更远”
assert receptive_field([(3, 1), (2, 2), (3, 1), (2, 2)]) == 10
print("Lab 2 自检 B 通过：参数量和感受野正确。")

### 观察：局部连接和参数共享到底省了多少参数

把第一层 `1×28×28 → 16×28×28` 换成**全连接层**，每个输出单元连接所有 784 个输入像素，每个输出单元有自己的权重，参数量会是多少？下面的单元格把两种做法放在一起比较（代码已提供）。

- **局部连接**：卷积的每个输出单元只看一个 $3\times3$ 的小窗口。
- **参数共享**：同一个输出通道在所有位置使用同一个 $3\times3$ 卷积核。

In [ ]:
c_in, c_out, kh, kw, H, W = 1, 16, 3, 3, 28, 28
conv_params = conv_param_count(c_in, c_out, kh, kw)
fc_params = (c_in * H * W) * (c_out * H * W) + c_out * H * W   # 每个输出单元一个 bias 的全连接层
print(f"卷积层参数量（共享）：{conv_params:>12,}")
print(f"同输出形状的全连接层：{fc_params:>12,}   （约为卷积层的 {fc_params / conv_params:,.0f} 倍）")
print(f"卷积层每个输出单元连接的输入数：{kh * kw}；全连接层：{c_in * H * W}")

fig, ax = plt.subplots(figsize=(6, 3.2))
bars = ax.bar(["Conv 1→16, k=3\n(局部连接 + 参数共享)", "Linear 784→12544\n(全连接)"], [conv_params, fc_params], color=[SERIES_COLORS[0], SERIES_COLORS[1]])
ax.set_yscale("log"); ax.set_ylabel("参数量（对数坐标）")
for bar, v in zip(bars, [conv_params, fc_params]):
    ax.text(bar.get_x() + bar.get_width() / 2, v, f"{v:,}", ha="center", va="bottom")
ax.set_title("同样的输入输出形状，两种层的参数量")
fig.tight_layout(); plt.show()

### 观察：共享的卷积核会收到所有位置的梯度

卷积核里的每个权重 $K_{u,v}$ 在每个输出位置都参与了计算。因此对 `out.sum()` 求梯度时，$\partial\,\text{out.sum()}/\partial K_{u,v}$ 等于所有窗口里**第 $(u,v)$ 个位置的输入值之和**。下面的单元格用 autograd 验证这一点（代码已提供，依赖你实现的 `torch_conv_from_numpy`）。

In [ ]:
conv_g = torch_conv_from_numpy(K_edge, 0.0)
x_g = torch.tensor(X_hand, dtype=torch.float32)[None, None]
conv_g(x_g).sum().backward()
grad_w = conv_g.weight.grad[0, 0].numpy()

# 手算：每个核位置 (u, v) 的梯度 = 所有窗口中对应位置的输入之和
expected = np.zeros((3, 3))
for u in range(3):
    for v in range(3):
        expected[u, v] = X_hand[u : u + 2, v : v + 2].sum()   # 2x2 个输出位置，每个位置贡献一个输入值
np.testing.assert_allclose(grad_w, expected, atol=1e-5)
print("conv.weight.grad =\n", grad_w)
print("bias 的梯度 =", conv_g.bias.grad.item(), "（等于输出位置个数 4）")
assert conv_g.bias.grad.item() == 4.0

## Lab 3：用 PyTorch 搭建分类网络

**你要实现**：`SimpleCNN.__init__` 和 `SimpleCNN.forward`，结构见 4.6 的表。

| 属性名 | 层 |
| --- | --- |
| `self.conv1` | `nn.Conv2d(1, 16, kernel_size=3, padding=1)` |
| `self.pool1` | `nn.MaxPool2d(kernel_size=2, stride=2)` |
| `self.conv2` | `nn.Conv2d(16, 32, kernel_size=3, padding=1)` |
| `self.pool2` | `nn.MaxPool2d(kernel_size=2, stride=2)` |
| `self.fc1` | `nn.Linear(32 * 7 * 7, 128)` |
| `self.fc2` | `nn.Linear(128, 10)` |

`forward` 里的顺序：`conv1 → ReLU → pool1 → conv2 → ReLU → pool2 → flatten → fc1 → ReLU → fc2`，返回 `[B, 10]` 的 logits。

**边写边检查维度**：每一行之后在心里写出 shape（见 4.6）。写完运行下面的 shape 表，它会通过 forward hook 打印每层真实输出。

**提示**：

- `__init__` 第一行必须是 `super().__init__()`。
- 最后一层不加 Softmax。
- 展平时保留 batch 维：`x.flatten(1)`；不要写 `x.view(-1)`。

In [ ]:
class SimpleCNN(nn.Module):
    """Conv-Pool-Conv-Pool-FC-FC 的 MNIST 分类器。

    Input : [B, 1, 28, 28]
    Output: [B, 10] logits
    """

    def __init__(self) -> None:
        # TODO: 调用 super().__init__()，然后按上表创建 conv1, pool1, conv2, pool2, fc1, fc2。
        raise NotImplementedError("TODO Lab 3: implement SimpleCNN.__init__")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: 按文字里的顺序写前向，并在每一步想清楚 shape。
        raise NotImplementedError("TODO Lab 3: implement SimpleCNN.forward")

In [ ]:
def trace_shapes(model: nn.Module, x: torch.Tensor) -> list[tuple[str, tuple, int]]:
    """用 forward hook 记录每个子模块的输出 shape 和参数量（已提供）。"""
    records, handles = [], []
    for name, module in model.named_children():
        def hook(mod, inp, out, name=name):
            records.append((name, tuple(out.shape), sum(p.numel() for p in mod.parameters())))
        handles.append(module.register_forward_hook(hook))
        if name == "fc1":   # fc1 的输入就是 Flatten 之后的特征
            handles.append(module.register_forward_pre_hook(lambda mod, inp: records.append(("flatten (fc1 输入)", tuple(inp[0].shape), 0))))
    with torch.no_grad():
        model(x)
    for h in handles:
        h.remove()
    return records


torch.manual_seed(SEED)
model_check = SimpleCNN()
dummy = torch.randn(4, 1, 28, 28)
rows = trace_shapes(model_check, dummy)
print(f"{'层':<22}{'输出 shape':<22}{'参数量':>10}")
print(f"{'input':<22}{str(tuple(dummy.shape)):<22}{0:>10}")
for name, shape, n in rows:
    print(f"{name:<22}{str(shape):<22}{n:>10,}")
total = sum(p.numel() for p in model_check.parameters())
print(f"{'合计':<22}{'':<22}{total:>10,}")

In [ ]:
# Lab 3 自检：每层 shape、参数量和输出
shapes = {name: shape for name, shape, _ in trace_shapes(model_check, dummy)}
assert shapes["conv1"] == (4, 16, 28, 28), f"conv1 输出应为 (4, 16, 28, 28)，得到 {shapes['conv1']}"
assert shapes["pool1"] == (4, 16, 14, 14), f"pool1 输出应为 (4, 16, 14, 14)，得到 {shapes['pool1']}"
assert shapes["conv2"] == (4, 32, 14, 14), f"conv2 输出应为 (4, 32, 14, 14)，得到 {shapes['conv2']}"
assert shapes["pool2"] == (4, 32, 7, 7), f"pool2 输出应为 (4, 32, 7, 7)，得到 {shapes['pool2']}"
assert shapes["flatten (fc1 输入)"] == (4, 1568), "展平后应为 (4, 1568)，别把 batch 维也展平了"
assert shapes["fc1"] == (4, 128)
assert shapes["fc2"] == (4, 10)

counts = {name: n for name, _, n in trace_shapes(model_check, dummy)}
assert counts["conv1"] == 160 and counts["conv2"] == 4640 and counts["fc1"] == 200832 and counts["fc2"] == 1290
assert total == 206922, f"总参数量应为 206922，得到 {total}"

with torch.no_grad():
    logits = model_check(dummy)
assert logits.shape == (4, 10) and logits.dtype == torch.float32
# 输出是 logits，不是概率：行和一般不等于 1
assert not torch.allclose(logits.softmax(1), logits, atol=1e-3)
assert not torch.allclose(logits.sum(1), torch.ones(4), atol=1e-3), "输出不应该已经做过 Softmax"
# 初始化时 10 类应接近均匀：loss 约 ln(10) = 2.30
init_loss = F.cross_entropy(logits, torch.randint(0, 10, (4,)))
assert 1.5 < init_loss.item() < 3.5, f"初始 loss 应接近 ln(10)≈2.30，得到 {init_loss.item():.3f}"
# ReLU 检查：fc1 和 fc2 的输入都来自 ReLU（再经过 MaxPool），必须非负；少了 ReLU 这里会出现负数
relu_inputs = {}
hs = [getattr(model_check, n).register_forward_pre_hook(lambda mod, inp, n=n: relu_inputs.__setitem__(n, inp[0].detach())) for n in ("fc1", "fc2")]
model_check(dummy)
for h_ in hs:
    h_.remove()
assert (relu_inputs["fc1"] >= 0).all(), "conv2 / pool2 之后应该有 ReLU（fc1 的输入不应有负数）"
assert (relu_inputs["fc2"] >= 0).all(), "fc1 之后应该有 ReLU（fc2 的输入不应有负数）"
assert (relu_inputs["fc1"] == 0).any() and (relu_inputs["fc2"] == 0).any(), "ReLU 应该把一部分值变成 0"
print("Lab 3 SimpleCNN 自检通过。初始 loss =", round(init_loss.item(), 3))

### 梯度检查

反向传播由 autograd 完成，但你仍然应该确认：**每个参数都收到了梯度，而且梯度是对的**。下面的单元格（已提供）做两件事：

1. 一次 `backward()` 之后，所有参数的 `.grad` 都不是 `None`、没有 NaN。
2. 用 **float64** 的中心差分检查几个权重的解析梯度（和 two-layer NN 实验里的 gradient check 是同一个思路）：

$$
\frac{\partial\mathcal L}{\partial\theta}\approx\frac{\mathcal L(\theta+\varepsilon)-\mathcal L(\theta-\varepsilon)}{2\varepsilon}
$$

In [ ]:
torch.manual_seed(SEED)
gc_model = SimpleCNN().double()
gx = torch.randn(6, 1, 28, 28, dtype=torch.float64)
gy = torch.randint(0, 10, (6,))
loss_fn_gc = nn.CrossEntropyLoss()

gc_model.zero_grad()
loss_fn_gc(gc_model(gx), gy).backward()
for name, p in gc_model.named_parameters():
    assert p.grad is not None, f"{name} 没有梯度：它可能没有被注册，或没有参与 forward"
    assert torch.isfinite(p.grad).all(), f"{name} 的梯度里有 NaN / Inf"

def numeric_grad(param, index, eps=1e-6):
    with torch.no_grad():
        old = param[index].item()
        param[index] = old + eps; plus = loss_fn_gc(gc_model(gx), gy).item()
        param[index] = old - eps; minus = loss_fn_gc(gc_model(gx), gy).item()
        param[index] = old
    return (plus - minus) / (2 * eps)

g = torch.Generator().manual_seed(0)
worst = 0.0
for name in ["conv1.weight", "conv1.bias", "conv2.weight", "fc1.weight", "fc2.weight", "fc2.bias"]:
    param = dict(gc_model.named_parameters())[name]
    for _ in range(4):
        index = tuple(int(torch.randint(0, s, (1,), generator=g)) for s in param.shape)
        analytic, numeric = param.grad[index].item(), numeric_grad(param, index)
        rel = abs(analytic - numeric) / max(1e-8, abs(analytic) + abs(numeric))
        worst = max(worst, rel)
        assert rel < 1e-5, f"{name}{index} 解析梯度 {analytic:.3e} 与数值梯度 {numeric:.3e} 不一致"
print(f"梯度检查通过，最大相对误差 {worst:.2e}")

## Lab 4：数据加载、训练、验证、保存和测试

**你要实现**：

| 函数 | 作用 |
| --- | --- |
| `make_dataloaders(data_dir, batch_size, seed)` | 返回 `train_loader, val_loader, test_loader` |
| `train_one_epoch(model, loader, loss_fn, optimizer, device)` | 训练一个 epoch，返回 `(平均 loss, accuracy)` |
| `evaluate(model, loader, loss_fn, device)` | 评估，返回 `(平均 loss, accuracy)` |
| `save_checkpoint(model, path)` / `load_checkpoint(model, path, device)` | 保存和恢复 `state_dict` |
| `fit(model, train_loader, val_loader, epochs, lr, device, checkpoint_path)` | 完整训练：每个 epoch 训练 + 验证，保存验证准确率最高的权重，返回历史记录 |

### 4.A 数据

- 变换：`ToTensor()` 再 `Normalize((MNIST_MEAN,), (MNIST_STD,))`。
- 划分：官方训练集用 `random_split(full, [N_TRAIN, N_VAL], generator=torch.Generator().manual_seed(seed))`。
- 官方测试集（`train=False`）**不参与训练和模型选择**。
- 训练 loader `shuffle=True`，验证和测试 loader `shuffle=False`。

### 4.B 训练一个 epoch（顺序不能乱）

```
model.train()
for images, labels in loader:
    images, labels = images.to(device), labels.to(device)
    logits = model(images)               # [B, 10]
    loss = loss_fn(logits, labels)       # labels: [B], int64
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
```

- 累计 loss 时要按 batch 大小加权：`loss.item() * images.size(0)`，最后除以样本总数。最后一个 batch 可能比 64 小，直接平均各 batch 的 loss 会有偏差。
- 准确率：`(logits.argmax(dim=1) == labels).sum().item()`，最后除以样本总数。

### 4.C 评估

- `model.eval()` 并用 `with torch.no_grad():` 包住，评估不能更新参数，也不需要计算图。
- 注意：评估函数结束后不要自动切回 `train()`，下一轮训练里的 `train_one_epoch` 自己会调用 `model.train()`。

### 4.D 保存与恢复

- 保存：`torch.save(model.state_dict(), path)`。
- 恢复：`state = torch.load(path, map_location=device, weights_only=True)`，然后 `model.load_state_dict(state)`。

In [ ]:
def make_dataloaders(
    data_dir: str = DATA_DIR,
    batch_size: int = BATCH_SIZE,
    seed: int = SEED,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    """构建 MNIST 的训练 / 验证 / 测试 DataLoader。

    Returns
    -------
    train_loader : 55,000 张，shuffle=True
    val_loader   : 5,000 张，shuffle=False
    test_loader  : 10,000 张，shuffle=False

    Notes
    -----
    图片经过 ToTensor + Normalize((0.1307,), (0.3081,))。
    训练 / 验证划分必须由带 seed 的 torch.Generator 控制，保证可复现。
    """
    # TODO: transform = Compose([ToTensor(), Normalize(...)])
    # TODO: full = datasets.MNIST(data_dir, train=True, download=True, transform=transform)
    #       test = datasets.MNIST(data_dir, train=False, download=True, transform=transform)
    # TODO: random_split 划分 full，创建三个 DataLoader。
    raise NotImplementedError("TODO Lab 4: implement make_dataloaders")

In [ ]:
# Lab 4 自检 A：数据加载（首次运行会下载 MNIST）
train_loader, val_loader, test_loader = make_dataloaders()
assert len(train_loader.dataset) == 55_000 and len(val_loader.dataset) == 5_000 and len(test_loader.dataset) == 10_000
images, labels = next(iter(train_loader))
assert images.shape == (64, 1, 28, 28) and images.dtype == torch.float32, f"图片 {images.shape} {images.dtype}"
assert labels.shape == (64,) and labels.dtype == torch.int64, f"标签 {labels.shape} {labels.dtype}"
assert 0 <= labels.min() and labels.max() <= 9
# 已标准化：背景像素 (0 - 0.1307) / 0.3081 ≈ -0.424，笔画最亮处 (1 - 0.1307) / 0.3081 ≈ 2.82
assert abs(images.min().item() - (-0.4242)) < 1e-3 and abs(images.max().item() - 2.8215) < 1e-3, "像素没有按 0.1307 / 0.3081 标准化"
# 训练集和验证集不能重叠；训练集要打乱，验证 / 测试集不能打乱
train_idx, val_idx = set(train_loader.dataset.indices), set(val_loader.dataset.indices)
assert not (train_idx & val_idx), "训练集和验证集有重叠样本"
assert len(train_idx | val_idx) == 60_000
from torch.utils.data import RandomSampler, SequentialSampler
assert isinstance(train_loader.sampler, RandomSampler), "训练 loader 应该 shuffle=True"
assert isinstance(val_loader.sampler, SequentialSampler) and isinstance(test_loader.sampler, SequentialSampler)
# 同一个 seed 得到同一个划分
_, val_loader_2, _ = make_dataloaders()
assert list(val_loader_2.dataset.indices) == list(val_loader.dataset.indices), "同一个 seed 的划分应该一致"
print("Lab 4 自检 A 通过：数据 55000 / 5000 / 10000，shape、dtype、标准化和划分都正确。")

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    loss_fn: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> tuple[float, float]:
    """训练一个 epoch。

    Returns
    -------
    avg_loss : float
        按样本数加权的平均 loss。
    accuracy : float
        训练过程中累计的准确率（0 到 1）。
    """
    # TODO: model.train()；遍历 loader；把 images、labels 移到 device；
    #       前向 → loss → zero_grad → backward → step；累计 loss 和正确数。
    raise NotImplementedError("TODO Lab 4: implement train_one_epoch")

In [ ]:
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    loss_fn: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    """在 loader 上评估，不更新参数。

    Returns
    -------
    avg_loss : float
    accuracy : float
    """
    # TODO: model.eval()；用 torch.no_grad() 包住循环；累计 loss 和正确数。
    raise NotImplementedError("TODO Lab 4: implement evaluate")

In [ ]:
def save_checkpoint(model: nn.Module, path: str) -> None:
    """把 model.state_dict() 保存到 path。"""
    # TODO: torch.save(...)
    raise NotImplementedError("TODO Lab 4: implement save_checkpoint")

In [ ]:
def load_checkpoint(model: nn.Module, path: str, device: torch.device) -> nn.Module:
    """把 path 里的权重读进 model，并把 model 放到 device 上，返回 model。"""
    # TODO: torch.load(path, map_location=device, weights_only=True)；load_state_dict；model.to(device)。
    raise NotImplementedError("TODO Lab 4: implement load_checkpoint")

In [ ]:
def fit(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int,
    lr: float,
    device: torch.device,
    checkpoint_path: str,
) -> dict[str, list[float]]:
    """完整训练流程。

    - 把 model 移到 device，创建 CrossEntropyLoss 和 Adam(lr)。
    - 每个 epoch：train_one_epoch，再 evaluate 验证集，打印一行日志。
    - 验证准确率创新高时 save_checkpoint。

    Returns
    -------
    history : dict
        键 "train_loss", "train_acc", "val_loss", "val_acc"，每个值是长度为 epochs 的 list。
    """
    # TODO: 见上面的说明。history 字典用 4 个空 list 初始化，每个 epoch 末尾 append。
    raise NotImplementedError("TODO Lab 4: implement fit")

In [ ]:
# Lab 4 自检 B：用一个小的合成数据集检查训练 / 评估 / 保存的行为（不依赖 MNIST）
import copy, os, tempfile
from torch.utils.data import TensorDataset

g = torch.Generator().manual_seed(0)
toy_x = torch.randn(256, 1, 28, 28, generator=g)
toy_y = (toy_x[:, 0, :14].mean(dim=(1, 2)) > toy_x[:, 0, 14:].mean(dim=(1, 2))).long()   # 上半 vs 下半的亮度，2 类
toy_loader = DataLoader(TensorDataset(toy_x, toy_y), batch_size=64, shuffle=True)

torch.manual_seed(SEED)
toy_model = SimpleCNN()
toy_loss_fn = nn.CrossEntropyLoss()
toy_opt = torch.optim.Adam(toy_model.parameters(), lr=1e-3)
cpu = torch.device("cpu")

# (1) train_one_epoch 会更新参数、返回 (loss, acc)、并让模型处在 train 模式
before = copy.deepcopy(toy_model.state_dict())
loss0, acc0 = train_one_epoch(toy_model, toy_loader, toy_loss_fn, toy_opt, cpu)
assert toy_model.training, "train_one_epoch 里应该调用 model.train()"
assert any(not torch.equal(before[k], v) for k, v in toy_model.state_dict().items()), "参数一次都没有更新：检查 optimizer.step()"
assert 0.0 <= acc0 <= 1.0 and loss0 > 0

# (2) evaluate 不改参数、不留梯度、处在 eval 模式；重复调用结果相同
for p in toy_model.parameters():
    p.grad = None
snapshot = copy.deepcopy(toy_model.state_dict())
e1 = evaluate(toy_model, toy_loader, toy_loss_fn, cpu)
e2 = evaluate(toy_model, toy_loader, toy_loss_fn, cpu)
assert not toy_model.training, "evaluate 里应该调用 model.eval()"
assert all(torch.equal(snapshot[k], v) for k, v in toy_model.state_dict().items()), "evaluate 不能改参数"
assert all(p.grad is None for p in toy_model.parameters()), "evaluate 应该在 torch.no_grad() 里，不产生梯度"
assert e1 == e2

# (3) 多训练几个 epoch，loss 应该明显下降；loss 按样本数加权（最后一个 batch 不满 64 个也要对，见下面的 odd_loader）
for _ in range(8):
    last_loss, last_acc = train_one_epoch(toy_model, toy_loader, toy_loss_fn, toy_opt, cpu)
assert last_loss < loss0 * 0.5, f"8 个 epoch 后 loss 应该明显下降：{loss0:.3f} → {last_loss:.3f}"
odd_loader = DataLoader(TensorDataset(toy_x[:100], toy_y[:100]), batch_size=64)   # 最后一个 batch 只有 36 个
l_odd, a_odd = evaluate(toy_model, odd_loader, toy_loss_fn, cpu)
with torch.no_grad():
    toy_model.eval()
    lg = toy_model(toy_x[:100])
np.testing.assert_allclose(l_odd, F.cross_entropy(lg, toy_y[:100]).item(), atol=1e-5, err_msg="loss 要按样本数加权，而不是直接平均各 batch")
np.testing.assert_allclose(a_odd, (lg.argmax(1) == toy_y[:100]).float().mean().item(), atol=1e-9)

# (4) 保存 / 恢复：恢复后的模型输出和原模型完全一致
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "toy.pt")
    save_checkpoint(toy_model, path)
    restored = load_checkpoint(SimpleCNN(), path, cpu)
    toy_model.eval(); restored.eval()
    with torch.no_grad():
        np.testing.assert_allclose(restored(toy_x[:8]).numpy(), toy_model(toy_x[:8]).numpy(), atol=1e-6)
# (5) fit：history 有 4 个键、长度等于 epochs；磁盘上的检查点是验证准确率最高的那个 epoch，而不是最后一个
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "fit.pt")
    torch.manual_seed(SEED)
    fit_model = SimpleCNN()
    val_loader_toy = DataLoader(TensorDataset(toy_x[:128], toy_y[:128]), batch_size=64)
    hist = fit(fit_model, toy_loader, val_loader_toy, 4, 3e-3, cpu, path)
    assert set(hist) == {"train_loss", "train_acc", "val_loss", "val_acc"}, f"history 的键不对：{set(hist)}"
    assert all(len(v) == 4 for v in hist.values()), "每个键应该有 epochs 个值"
    assert os.path.exists(path), "fit 应该保存检查点"
    best = load_checkpoint(SimpleCNN(), path, cpu)
    _, best_acc = evaluate(best, val_loader_toy, toy_loss_fn, cpu)
    np.testing.assert_allclose(best_acc, max(hist["val_acc"]), atol=1e-9, err_msg="检查点应该是验证准确率最高的那个 epoch")
print(f"Lab 4 自检 B 通过：训练 / 评估 / 保存 / fit 行为正确（loss {loss0:.3f} → {last_loss:.3f}）。")

### 验收 1：先让模型记住固定的 32 张训练图

在训练全量数据之前，先做这个检查。把**同样的 32 张图**反复喂给模型，一个正确的网络、loss、梯度和优化器组合应该能把它们全部记住（准确率 100%，loss 接近 0）。如果记不住，说明某一步有 bug，这时候训练 55,000 张只会浪费时间。

如果这个检查失败，按下面的顺序排查：

1. Lab 3 的 shape 表和梯度检查是否通过？
2. `loss_fn` 的输入是不是 logits（模型里有没有多加 Softmax）？
3. `optimizer.zero_grad()`、`loss.backward()`、`optimizer.step()` 是不是都调用了，顺序对不对？
4. `labels` 的 dtype 是不是 `int64`，shape 是不是 `[B]`？

下面的单元格已提供。

In [ ]:
# 验收 1：过拟合固定的 32 张训练图
fixed_subset = Subset(train_loader.dataset, list(range(32)))
fixed_loader = DataLoader(fixed_subset, batch_size=32, shuffle=False)

torch.manual_seed(SEED)
mem_model = SimpleCNN().to(DEVICE)
mem_loss_fn = nn.CrossEntropyLoss()
mem_opt = torch.optim.Adam(mem_model.parameters(), lr=LEARNING_RATE)

mem_history = []
for step in range(1, 151):
    loss_v, acc_v = train_one_epoch(mem_model, fixed_loader, mem_loss_fn, mem_opt, DEVICE)
    mem_history.append((loss_v, acc_v))
    if step in (1, 10, 25, 50, 100, 150):
        print(f"step {step:>3}: loss {loss_v:.4f}  acc {acc_v:.3f}")

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot([h[0] for h in mem_history], color=SERIES_COLORS[0], label="loss")
ax.set_yscale("log"); ax.set_xlabel("step"); ax.set_ylabel("loss（对数坐标）")
ax.set_title("固定 32 张图：loss 应该一路降到接近 0")
fig.tight_layout(); plt.show()

assert mem_history[-1][1] == 1.0, f"32 张图的准确率应该达到 100%，得到 {mem_history[-1][1]:.3f}"
assert mem_history[-1][0] < 0.05, f"loss 应该小于 0.05，得到 {mem_history[-1][0]:.4f}"
print("验收 1 通过：模型能记住固定的 32 张图。")

### 验收 2：训练完整数据

先训练 5 个 epoch，按验证集准确率保存最好的权重。第一次运行大约需要几分钟，取决于你的设备。下面的单元格已提供，调用你实现的 `fit`。

In [ ]:
torch.manual_seed(SEED)
model = SimpleCNN()
history = fit(model, train_loader, val_loader, EPOCHS, LEARNING_RATE, DEVICE, CHECKPOINT_PATH)
assert history["val_acc"][-1] > 0.975, f"验证准确率 {history['val_acc'][-1]:.4f} 偏低，检查训练流程"
print("最好的验证准确率：", max(history["val_acc"]))

In [ ]:
# 观察图：训练 / 验证曲线
epochs_axis = np.arange(1, len(history["train_loss"]) + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
a1.plot(epochs_axis, history["train_loss"], "-o", color=SERIES_COLORS[0], label="train")
a1.plot(epochs_axis, history["val_loss"], "--s", color=SERIES_COLORS[1], label="validation")
a1.set_xlabel("epoch"); a1.set_ylabel("loss"); a1.set_title("Loss"); a1.legend()
a2.plot(epochs_axis, history["train_acc"], "-o", color=SERIES_COLORS[0], label="train")
a2.plot(epochs_axis, history["val_acc"], "--s", color=SERIES_COLORS[1], label="validation")
a2.set_xlabel("epoch"); a2.set_ylabel("accuracy"); a2.set_title("Accuracy"); a2.legend()
for a in (a1, a2):
    a.set_xticks(epochs_axis); a.grid(alpha=0.25)
fig.tight_layout(); plt.show()

### 验收 3：加载最好的权重，在测试集上评估

**只评估一次**。测试集不能用来调超参数或挑选 epoch，否则测试准确率就不再是对新数据表现的无偏估计。

In [ ]:
best_model = load_checkpoint(SimpleCNN(), CHECKPOINT_PATH, DEVICE)
test_loss, test_acc = evaluate(best_model, test_loader, nn.CrossEntropyLoss(), DEVICE)
print(f"Test loss {test_loss:.4f}   Test accuracy {test_acc:.4f}")
assert test_acc > 0.975, f"测试准确率 {test_acc:.4f}，目标是 98% 左右"
print("验收 3 通过：测试准确率达标。")

> **TODO（研究）**：
> 1. 在上面的训练曲线里，train accuracy 和 validation accuracy 之间有没有差距？为什么用 validation 选 epoch，而不是用 test？
> 2. 把 `BATCH_SIZE` 分别改成 16 / 256，把 `LEARNING_RATE` 改成 0.01 / 0.0001，各训练 5 个 epoch。记录验证准确率，写下你观察到的规律。
> 3. 训练前的 `torch.manual_seed(SEED)` 固定了初始化，但在 GPU / MPS 上结果仍然可能有细微差别。查一下 `torch.use_deterministic_algorithms` 的文档。

## Lab 5：看第一层的 feature maps，和分类错误的图片

**你要实现**：

- `get_first_layer_feature_maps(model, image)`：`image` 是 `[1, 28, 28]` 的 Tensor（已标准化）。返回第一层 `conv1` **加上 ReLU 之后**的输出，形状 `[16, 28, 28]`，放在 CPU 上，不带梯度。
- `get_misclassified(model, loader, device, max_n=16)`：遍历 `loader`，收集前 `max_n` 张被分错的图片。返回字典：

| 键 | 类型 | Shape |
| --- | --- | --- |
| `"images"` | `float32` Tensor | `[n, 1, 28, 28]`，CPU |
| `"y_true"` | `int64` Tensor | `[n]` |
| `"y_pred"` | `int64` Tensor | `[n]` |
| `"confidence"` | `float32` Tensor | `[n]`，预测类别的 Softmax 概率 |

**提示**：

- 两个函数都要 `model.eval()` 和 `torch.no_grad()`。`model` 在 `device` 上，输入要先 `.to(device)`，返回前用 `.cpu()`。
- `get_first_layer_feature_maps` 里，给 `image` 加 batch 维用 `image.unsqueeze(0)`，输出去掉 batch 维用 `[0]`。
- 置信度用 `torch.softmax(logits, dim=1)` 再取 `max`。注意：Softmax 只在这里用来**解释**输出，训练时不用。

In [ ]:
def get_first_layer_feature_maps(model: SimpleCNN, image: torch.Tensor) -> torch.Tensor:
    """返回 conv1 + ReLU 的输出。

    Parameters
    ----------
    image : Tensor, shape (1, 28, 28)

    Returns
    -------
    Tensor, shape (16, 28, 28)，在 CPU 上，不带梯度。
    """
    # TODO: 找到 model 所在的 device（next(model.parameters()).device）；
    #       eval + no_grad；F.relu(model.conv1(image 加 batch 维))；去掉 batch 维，.cpu()。
    raise NotImplementedError("TODO Lab 5: implement get_first_layer_feature_maps")

In [ ]:
def get_misclassified(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    max_n: int = 16,
) -> dict[str, torch.Tensor]:
    """收集 loader 里前 max_n 张被分错的图片。

    Returns
    -------
    dict
        键 "images" [n,1,28,28]、"y_true" [n]、"y_pred" [n]、"confidence" [n]，都在 CPU 上。
        不足 max_n 张时返回实际数量。
    """
    # TODO: eval + no_grad；每个 batch 算 logits、pred、概率；
    #       用布尔掩码 wrong = pred != labels 选出错误样本，累计到 max_n 张就可以停。
    raise NotImplementedError("TODO Lab 5: implement get_misclassified")

In [ ]:
# Lab 5 自检
sample_img, sample_label = test_loader.dataset[0]
fmaps = get_first_layer_feature_maps(best_model, sample_img)
assert fmaps.shape == (16, 28, 28), f"feature maps 应为 (16, 28, 28)，得到 {tuple(fmaps.shape)}"
assert fmaps.device.type == "cpu" and not fmaps.requires_grad
assert (fmaps >= 0).all(), "应该在 ReLU 之后返回"
with torch.no_grad():
    manual = F.relu(best_model.conv1(sample_img.unsqueeze(0).to(DEVICE)))[0].cpu()
torch.testing.assert_close(fmaps, manual)

wrong = get_misclassified(best_model, test_loader, DEVICE, max_n=16)
n = len(wrong["y_true"])
assert 1 <= n <= 16, f"测试准确率不到 100%，应该能找到错误样本（得到 {n} 张）"
assert wrong["images"].shape == (n, 1, 28, 28) and wrong["images"].device.type == "cpu"
assert wrong["y_true"].dtype == torch.int64 and wrong["y_pred"].dtype == torch.int64
assert (wrong["y_true"] != wrong["y_pred"]).all(), "返回的应该全是分类错误的样本"
with torch.no_grad():
    expected_conf = torch.softmax(best_model(wrong["images"].to(DEVICE)), dim=1).max(dim=1).values.cpu()
torch.testing.assert_close(wrong["confidence"], expected_conf, atol=1e-5, rtol=1e-4)
# 用模型重新预测，验证 y_pred 确实是模型的输出
with torch.no_grad():
    best_model.eval()
    again = best_model(wrong["images"].to(DEVICE)).argmax(1).cpu()
assert torch.equal(again, wrong["y_pred"])
print(f"Lab 5 自检通过，找到 {n} 张错误样本。")

In [ ]:
# 观察图 1：第一层的 16 个卷积核，以及一张输入图经过它们之后的 feature maps
def denormalize(img):
    return img * MNIST_STD + MNIST_MEAN

kernels = best_model.conv1.weight.detach().cpu()[:, 0]          # [16, 3, 3]
fig = plt.figure(figsize=(12, 7.2))
gs = fig.add_gridspec(5, 8, height_ratios=[1, 1, 0.15, 1, 1])
for idx in range(16):
    ax = fig.add_subplot(gs[idx // 8, idx % 8])
    vmax = kernels.abs().max().item()
    ax.imshow(kernels[idx], cmap="RdBu", vmin=-vmax, vmax=vmax)
    ax.set_title(f"核 {idx}", fontsize=8); ax.set_xticks([]); ax.set_yticks([])
fig.text(0.5, 0.955, "conv1 的 16 个 3×3 卷积核（红 = 正权重，蓝 = 负权重）", ha="center", fontsize=11)

for idx in range(16):
    ax = fig.add_subplot(gs[3 + idx // 8, idx % 8])
    ax.imshow(fmaps[idx], cmap=SEQ_CMAP)
    ax.set_title(f"map {idx}", fontsize=8); ax.set_xticks([]); ax.set_yticks([])
fig.text(0.5, 0.485, f"这张输入图（标签 {sample_label}）经过 conv1 + ReLU 的 16 张 feature maps（每张图独立缩放颜色）", ha="center", fontsize=11)
fig.subplots_adjust(top=0.93, bottom=0.03, hspace=0.45)
plt.show()

fig, ax = plt.subplots(figsize=(2.6, 2.6))
ax.imshow(denormalize(sample_img[0]), cmap="gray"); ax.set_title(f"输入图（标签 {sample_label}）"); ax.axis("off")
plt.show()

In [ ]:
# 观察图 2：分类错误的图片（标题：T = 真实标签，P = 模型预测，conf = 模型对预测的置信度）
n = len(wrong["y_true"])
cols = 8
rows_n = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows_n, cols, figsize=(1.9 * cols, 2.9 * rows_n))
axes = np.atleast_1d(axes).ravel()
for ax in axes:
    ax.axis("off")
for k in range(n):
    axes[k].imshow(denormalize(wrong["images"][k, 0]), cmap="gray")
    axes[k].set_title(f"T={wrong['y_true'][k].item()}  P={wrong['y_pred'][k].item()}\nconf {wrong['confidence'][k].item():.2f}", fontsize=8, color=HILITE)
fig.suptitle("前 16 张分类错误的测试图片", y=1.0)
fig.tight_layout(h_pad=2.0); plt.show()

In [ ]:
# 观察图 3：混淆矩阵（行 = 真实类别，列 = 预测类别），对角线以外是错误
all_pred, all_true = [], []
best_model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        all_pred.append(best_model(images.to(DEVICE)).argmax(1).cpu()); all_true.append(labels)
all_pred, all_true = torch.cat(all_pred), torch.cat(all_true)
confusion = torch.bincount(all_true * 10 + all_pred, minlength=100).reshape(10, 10).numpy()

fig, ax = plt.subplots(figsize=(6.2, 5.4))
off_diag = confusion.copy(); np.fill_diagonal(off_diag, 0)
im = ax.imshow(off_diag, cmap=SEQ_CMAP)
for (r, c), v in np.ndenumerate(confusion):
    ax.text(c, r, str(v), ha="center", va="center", fontsize=8, color="#bbb" if r == c else ("white" if off_diag[r, c] > off_diag.max() * 0.6 else "#222"))
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("预测类别"); ax.set_ylabel("真实类别")
ax.set_title("测试集混淆矩阵（颜色只表示错误，灰色是对角线上的正确数）")
fig.colorbar(im, ax=ax, label="错误个数"); fig.tight_layout(); plt.show()
worst_pair = np.unravel_index(off_diag.argmax(), off_diag.shape)
print(f"最常见的错误：真实 {worst_pair[0]} 被预测成 {worst_pair[1]}，共 {off_diag.max()} 次")

> **TODO（研究）**：
> 1. 看上面的卷积核和 feature maps：哪些核像是在检测边缘、横线或竖线？哪些 feature map 几乎全是 0（所谓“死亡”通道）？写下你的观察。
> 2. 看分类错误的图片：这些图片人眼能认出来吗？模型对错误预测的置信度高不高？这说明 Softmax 概率和“模型真的懂了”之间有什么区别？
> 3. 第二层 `conv2` 的卷积核是 `[32, 16, 3, 3]`，怎么可视化？提示：它的每个核有 16 个输入通道。可以先对输入通道取平均。
> 4. 尝试 **Grad-CAM**：用最后一个卷积层的 feature maps 和类别分数的梯度，看模型在“看”图片的哪个区域。

## Lab 6：NumPy 手写 vs PyTorch，速度与实现的对比

下面的单元格（已提供）在同一张 $28\times28$ 图片上用 NumPy 版和 PyTorch 版做同样的卷积，并计时。它会调用 Lab 1 里的进阶函数 `conv2d_multi_channel`，如果你跳过了它，这个 Lab 先用单通道的 `conv2d_forward` 循环 16 次替代。重点不是绝对数字，而是它们差几个数量级，以及差距是怎么来的。

In [ ]:
x28 = rng.standard_normal((1, 28, 28)).astype(np.float64)
w16 = rng.standard_normal((16, 1, 3, 3)); b16 = rng.standard_normal(16)

t0 = time.perf_counter()
out_np = conv2d_multi_channel(x28, w16, b16, stride=1, padding=1)
t_np = time.perf_counter() - t0

conv_ref = nn.Conv2d(1, 16, 3, padding=1).double()
with torch.no_grad():
    conv_ref.weight.copy_(torch.from_numpy(w16)); conv_ref.bias.copy_(torch.from_numpy(b16))
xt = torch.from_numpy(x28)[None]
conv_ref(xt)    # 预热
t0 = time.perf_counter()
for _ in range(100):
    out_t = conv_ref(xt)
t_torch = (time.perf_counter() - t0) / 100
np.testing.assert_allclose(out_np, out_t[0].detach().numpy(), atol=1e-10)

print(f"NumPy 手写（1×28×28 → 16×28×28）：{t_np * 1e3:8.2f} ms")
print(f"nn.Conv2d（CPU，float64）       ：{t_torch * 1e3:8.3f} ms")
print(f"差距约 {t_np / t_torch:,.0f} 倍")
print(f"\n如果用 NumPy 版训练：只算第一层卷积的前向，55,000 张 × 5 个 epoch 就要约 {t_np * 55_000 * 5 / 3600:.0f} 小时（还没算第二层和反向传播）。")

## 进阶研究清单

完成上面的 Lab 之后，可以按兴趣继续。每一项都可以单独做成一个小实验。

> **TODO（研究）**
>
> - [ ] **Batch Normalization**：在每个 `Conv` 后加 `nn.BatchNorm2d`，对比收敛速度。理解 `model.train()` 与 `model.eval()` 下 BatchNorm 行为的区别。
> - [ ] **Dropout**：在 `fc1` 后加 `nn.Dropout(0.5)`，观察 train / val 差距的变化。
> - [ ] **数据增强**：训练集加 `RandomAffine`（小角度旋转、平移），测试集不加。对比准确率和训练曲线。
> - [ ] **1×1 卷积与 Global Average Pooling**：用 GAP 替换 `Flatten + fc1`，参数量能减少多少？准确率变化如何？
> - [ ] **感受野**：画出 `pool2` 输出（`fc1` 之前）里某个位置在原图上对应的 $10\times10$ 区域，验证 `receptive_field` 的结果（`conv2` 输出的感受野是 8）。
> - [ ] **im2col 与卷积反向传播**：见 Lab 1 末尾的研究提示，自己写 `conv2d_backward` 并用 `torch.autograd` 校验。
> - [ ] **从 MNIST 到 CIFAR-10 / Fashion-MNIST**：输入通道变成 3 时哪些代码要改？准确率会掉多少，为什么？
> - [ ] **TensorFlow / Keras 对照**：见 2.4 的 TODO，用 Keras 写同一个网络并对比 `model.summary()`。
> - [ ] **阅读**：[PyTorch 官方分类教程](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)、LeNet-5（LeCun et al., 1998）、本仓库 `01_machine_learning/VGG/` 里的 VGG 练习。

## 记录你的观察

在这里写下你的结论（实验后填写）：

| 项目 | 你的结果 |
| --- | --- |
| 32 张图记住所需 step 数 | |
| 5 epoch 后验证准确率 | |
| 测试准确率 | |
| 最常见的错误类型 | |
| `receptive_field` 在 `fc1` 前的值（你的答案，不用查） | |